# 7.7 Cluster Evaluation Metrics

**Machine Learning Mastery: Mathematics to MLOps** · Chapter 7: Clustering Techniques

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RithikRaj129/ml-mastery/blob/main/chapter-07-clustering-techniques/07.07_cluster-evaluation-metrics.ipynb)

Companion code for Section 7.7 of the book. Run the cells in order from top to bottom.

**Example 7.7.1**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import (
    silhouette_score, silhouette_samples,
    davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score,
    homogeneity_score, completeness_score, v_measure_score,
    fowlkes_mallows_score
)
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)
X, y_true = make_blobs(n_samples=400, centers=4, cluster_std=0.8, random_state=42)
X = StandardScaler().fit_transform(X)

# ── Comprehensive metric comparison across algorithms ─────────────
algorithms = {
    'K-Means (k=4)':     KMeans(n_clusters=4, n_init=10, random_state=42),
    'K-Means (k=3)':     KMeans(n_clusters=3, n_init=10, random_state=42),
    'Agglomerative (k=4)':AgglomerativeClustering(n_clusters=4, linkage='ward'),
    'DBSCAN':             DBSCAN(eps=0.35, min_samples=5),
    'GMM (k=4)':          GaussianMixture(n_components=4, n_init=5, random_state=42),
}

print('=== Comprehensive Cluster Evaluation ===')
print(f'{"Algorithm":<22} {"Silhouette":>11} {"Davies-B":>10} {"CH":>8} {"ARI":>8} {"NMI":>8} {"V-meas":>8}')
print('='*80)

metric_results = {}
for name, alg in algorithms.items():
    labels = alg.fit_predict(X)
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    if n_clusters < 2:
        print(f'{name:<22}  (insufficient clusters)')
        continue

    mask = labels >= 0   # Exclude noise for DBSCAN
    sil  = silhouette_score(X[mask], labels[mask])  if mask.sum()>1 else 0
    db   = davies_bouldin_score(X[mask], labels[mask])
    ch   = calinski_harabasz_score(X[mask], labels[mask])
    ari  = adjusted_rand_score(y_true[mask], labels[mask])
    nmi  = normalized_mutual_info_score(y_true[mask], labels[mask])
    vm   = v_measure_score(y_true[mask], labels[mask])
    print(f'{name:<22} {sil:>11.4f} {db:>10.4f} {ch:>8.1f} {ari:>8.4f} {nmi:>8.4f} {vm:>8.4f}')
    metric_results[name] = {'sil':sil,'db':db,'ch':ch,'ari':ari,'nmi':nmi,'vm':vm}

# ── Silhouette analysis plot ──────────────────────────────────────
km_best = KMeans(n_clusters=4, n_init=10, random_state=42)
labels_best = km_best.fit_predict(X)
sil_vals = silhouette_samples(X, labels_best)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Cluster Evaluation - Silhouette Analysis', fontsize=14, fontweight='bold')

# Silhouette plot
ax = axes[0]
palette = ['#E53935','#1976D2','#43A047','#FF6F00']
y_lower = 10
for k in range(4):
    sil_k = np.sort(sil_vals[labels_best == k])
    size_k = sil_k.shape[0]
    y_upper = y_lower + size_k
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, sil_k,
                    facecolor=palette[k], edgecolor='white', alpha=0.9)
    ax.text(-0.05, y_lower + 0.5*size_k, f'C{k}', fontsize=11, fontweight='bold')
    y_lower = y_upper + 10

mean_sil = silhouette_score(X, labels_best)
ax.axvline(mean_sil, color='red', ls='--', lw=2, label=f'Mean sil={mean_sil:.3f}')
ax.set_xlabel('Silhouette Coefficient'); ax.set_ylabel('Cluster')
ax.set_title('Silhouette Plot (k=4)\n(Wide, right-shifted bars = good cluster)')
ax.legend(); ax.grid(alpha=0.3)

# Elbow + Silhouette + DB for k selection
ax = axes[1]
k_range = range(2, 9)
sil_k, db_k, ch_k = [], [], []
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    lbl = km.fit_predict(X)
    sil_k.append(silhouette_score(X, lbl))
    db_k.append(davies_bouldin_score(X, lbl))
    ch_k.append(calinski_harabasz_score(X, lbl))

ax2 = ax.twinx()
l1, = ax.plot(k_range, sil_k, 'b-o', ms=7, lw=2, label='Silhouette ↑')
l2, = ax.plot(k_range, db_k,  'r-o', ms=7, lw=2, label='Davies-Bouldin ↓')
l3, = ax2.plot(k_range, ch_k, 'g-o', ms=7, lw=2, label='Calinski-H ↑')
ax.axvline(x=4, color='black', ls='--', lw=2, label='Optimal k=4')
ax.set_xlabel('k (Number of Clusters)')
ax.set_ylabel('Silhouette / Davies-Bouldin')
ax2.set_ylabel('Calinski-Harabasz', color='green')
ax.set_title('All Three Internal Metrics vs k\n(All agree at k=4)')
lines = [l1, l2, l3]; labels = [l.get_label() for l in lines]
ax.legend(lines, labels, fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('cluster_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()